In [1]:
# --- Cell 1: Setup & Config ---

import os, re, time, json, math, textwrap, pathlib, itertools, collections, random
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Tuple, Optional, Set

import requests
from xml.etree import ElementTree as ET

import numpy as np

# Try optional deps; pipeline still runs without them.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SK_TFIDF_OK = True
except Exception:
    SK_TFIDF_OK = False

try:
    # Optional sentence embeddings (local model recommended).
    # If you don't have internet or cached models, set USE_EMBEDDINGS=False below.
    from sentence_transformers import SentenceTransformer
    SBERT_OK = True
except Exception:
    SBERT_OK = False

# -----------------------
# Global knobs
# -----------------------
EUTILS_BASE      = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
ENTREZ_EMAIL     = os.getenv("ENTREZ_EMAIL", "you@example.com")  # PATCH POINT: set
ENTREZ_API_KEY   = os.getenv("ENTREZ_API_KEY", "")               # PATCH POINT: set (optional)
HTTP_TIMEOUT     = 30
REQUEST_DELAY    = 0.34 if not ENTREZ_API_KEY else 0.11

UNIVERSE_FETCH_MAX = 1800          # budget of efetch records across selected queries
TARGET_MIN, TARGET_MAX = 150, 10000  # desired hit count window

# Ranking knobs
TOP_K_SCREEN      = 200            # how many to keep for triage table (after ranking)
USE_EMBEDDINGS    = False          # PATCH POINT: set True if you have sentence-transformers locally
EMBED_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"  # cached model path/name

# Snowballing knobs (ICite)
ICITE_BASE = "https://icite.od.nih.gov/api/pubs"
ICITE_SLEEP = 0.34
ICITE_HTTP_TIMEOUT = 30

# Output dir
OUT_DIR = pathlib.Path("triage_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(13)
np.random.seed(13)

# -----------------------
# Protocol schema
# -----------------------
@dataclass
class Protocol:
    narrative_question: str
    deterministic_filters: Dict[str, Any]  # {"languages":[...], "year_min": int}
    designs_allowed: List[str] = field(default_factory=list)  # user-allowed study designs
    publication_types_allowlist: List[str] = field(default_factory=list)
    publication_types_blocklist: List[str] = field(default_factory=list)

    # user-provided domain tokens (optional; if blank we'll mine from key articles)
    P_terms: List[str] = field(default_factory=list)
    I_terms: List[str] = field(default_factory=list)
    C_terms: List[str] = field(default_factory=list)
    O_terms: List[str] = field(default_factory=list)

    # which blocks are mandatory in the query logic
    mandatory_blocks: List[str] = field(default_factory=lambda: ["P","I"])  # PATCH POINT

    # your seed/key PMIDs (high-confidence)
    key_pmids: List[int] = field(default_factory=list)

    # optional: design filters for query (kept minimal: we'll mostly prefilter after fetch)
    use_pubtype_query_filter: bool = False

# Helper: load protocol JSON from disk or inline
def load_protocol(path_or_json: str) -> Protocol:
    if os.path.exists(path_or_json):
        obj = json.loads(pathlib.Path(path_or_json).read_text(encoding="utf-8"))
    else:
        obj = json.loads(path_or_json)
    return Protocol(**obj)

print("Cell 1 ready. Set ENTREZ_EMAIL/ENTREZ_API_KEY env vars as needed.")


Cell 1 ready. Set ENTREZ_EMAIL/ENTREZ_API_KEY env vars as needed.


In [ ]:
# --- Cell 2: PubMed E-utilities & MeSH mining ---

S = requests.Session()
S.headers.update({"User-Agent": f"sr-triage/1.0 ({ENTREZ_EMAIL})"})

def eutils_get(path: str, params: Dict[str, Any]) -> requests.Response:
    p = dict(params)
    p["email"] = ENTREZ_EMAIL
    if ENTREZ_API_KEY: p["api_key"] = ENTREZ_API_KEY
    url = f"{EUTILS_BASE.rstrip('/')}/{path.lstrip('/')}"
    r = S.get(url, params=p, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    return r

def esearch_count_and_ids(term: str, retmax: int = 3000) -> Tuple[int, List[str]]:
    params = {"db":"pubmed","retmode":"json","term":term,"retmax":retmax,"usehistory":"n"}
    r = eutils_get("esearch.fcgi", params); js = r.json().get("esearchresult",{})
    count = int(js.get("count","0"))
    ids = js.get("idlist",[])
    time.sleep(REQUEST_DELAY)
    return count, ids

def efetch_xml(pmids: List[str]) -> str:
    if not pmids: return ""
    ids = ",".join(str(x) for x in pmids)
    params = {"db":"pubmed","retmode":"xml","rettype":"abstract","id":ids}
    r = eutils_get("efetch.fcgi", params)
    time.sleep(REQUEST_DELAY)
    return r.text

def parse_pubmed_xml(xml_text: str) -> List[Dict[str, Any]]:
    out=[]
    if not xml_text.strip(): return out
    root=ET.fromstring(xml_text)
    def _join(node):
        if node is None: return ""
        try: return "".join(node.itertext())
        except Exception: return node.text or ""
    for art in root.findall(".//PubmedArticle"):
        pmid = art.findtext(".//PMID") or ""
        title = _join(art.find(".//ArticleTitle")).strip()
        abs_nodes = art.findall(".//Abstract/AbstractText")
        abstract = " ".join(_join(n).strip() for n in abs_nodes) if abs_nodes else ""
        # year
        year=None
        for path in (".//ArticleDate/Year",".//PubDate/Year",".//DateCreated/Year",".//PubDate/MedlineDate"):
            s=art.findtext(path)
            if s:
                m=re.search(r"\d{4}", s)
                if m: year=int(m.group(0)); break
        lang = art.findtext(".//Language") or None
        pubtypes = [pt.text for pt in art.findall(".//PublicationTypeList/PublicationType") if pt.text]
        mesh = [mh.findtext("./DescriptorName") for mh in art.findall(".//MeshHeadingList/MeshHeading") if mh.findtext("./DescriptorName")]
        first_author = None
        fa = art.find(".//AuthorList/Author[1]")
        if fa is not None:
            last = fa.findtext("./LastName") or ""
            init = fa.findtext("./Initials") or ""
            first_author = (last or "").strip() or None
        out.append({"pmid":pmid,"title":title,"abstract":abstract,"year":year,"language":lang,"pubtypes":pubtypes,"mesh":mesh,"first_author":first_author})
    return out

def fetch_records_by_pmids(pmids: List[int]) -> List[Dict[str,Any]]:
    pmids=[str(int(p)) for p in pmids if p]
    batch=300
    allrecs=[]
    for i in range(0, len(pmids), batch):
        xml = efetch_xml(pmids[i:i+batch])
        allrecs += parse_pubmed_xml(xml)
    return allrecs

# -----------------------
# MeSH mining from seeds
# -----------------------
def _normalize_token(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def mine_mesh_from_key_articles(seed_records: List[Dict[str,Any]]) -> List[str]:
    mesh_terms=[]
    for r in seed_records:
        mesh_terms += (r.get("mesh") or [])
    mesh_terms = [m for m in mesh_terms if m]
    mesh_terms = list(dict.fromkeys(_normalize_token(m) for m in mesh_terms))
    return mesh_terms

print("Cell 2 ready.")


In [ ]:
# --- Cell 3: Bin MeSH to P/I/C/O; merge with user tokens ---

def contains_any(hay: str, needles: List[str]) -> bool:
    hay = _normalize_token(hay)
    for n in needles:
        n = _normalize_token(n)
        if not n: continue
        if n in hay: return True
    return False

def bin_mesh_terms(mesh_terms: List[str], proto: Protocol) -> Dict[str, List[str]]:
    """
    Deterministic assignment:
    - If user provided P/I/C/O tokens, we use them to anchor binning of MeSH (by substring).
    - Otherwise, small fallback heuristics (domain-specific tokens you can extend at PATCH POINT).
    """
    P_seeds = [*proto.P_terms]
    I_seeds = [*proto.I_terms]
    C_seeds = [*proto.C_terms]
    O_seeds = [*proto.O_terms]

    # PATCH POINT: Add/adjust domain heuristics when user did not provide enough seeds
    if not P_seeds:
        P_seeds += ["pectus excavatum","nuss","mirpe","thoracic surgery","pediatric","adolescent","children"]
    if not I_seeds:
        I_seeds += ["cryoablation","cryoanalgesia","intercostal nerve"]
    # comparators and outcomes are very topic-dependent; keep conservative defaults when absent
    if not C_seeds:
        C_seeds += ["thoracic epidural","paravertebral block","intercostal nerve block","erector spinae plane block","multimodal analgesia"]
    if not O_seeds:
        O_seeds += ["pain","pain scores","opioid","length of stay","nausea","vomiting","complications"]

    P,I,C,O = [],[],[],[]
    for m in mesh_terms:
        if contains_any(m, I_seeds): I.append(m); continue
        if contains_any(m, C_seeds): C.append(m); continue
        if contains_any(m, O_seeds): O.append(m); continue
        if contains_any(m, P_seeds): P.append(m); continue
        # unassigned MeSH are ignored (or could go to G bucket if you want)

    def uniq_cap(lst, cap=12):
        seen=[]; 
        for x in lst:
            x=_normalize_token(x)
            if x and x not in seen:
                seen.append(x)
            if len(seen)>=cap: break
        return seen

    return {
        "P": uniq_cap(P or P_seeds, cap=12),
        "I": uniq_cap(I or I_seeds, cap=12),
        "C": uniq_cap(C or C_seeds, cap=12),
        "O": uniq_cap(O or O_seeds, cap=12),
    }

print("Cell 3 ready.")


In [ ]:
# --- Cell 4: Query grid & counts ---

def or_block(terms: List[str], field="tiab") -> str:
    toks=[]
    for t in terms:
        t=t.strip()
        if not t: continue
        if " " in t or "-" in t:
            toks.append(f"\"{t}\"[{field}]")
        else:
            toks.append(f"{t}[{field}]")
    if not toks: return ""
    return "(" + " OR ".join(toks) + ")"

def build_candidate_queries(bins: Dict[str,List[str]], proto: Protocol) -> List[Dict[str,Any]]:
    # Cap tokens to reduce explosions
    def cap(lst, n): return lst[:n] if n>0 else []

    P = cap(bins["P"], 8)
    I = cap(bins["I"], 8)
    C = cap(bins["C"], 8)
    O = cap(bins["O"], 8)

    Pq = or_block(P, "tiab"); Iq = or_block(I, "tiab")
    Cq = or_block(C, "tiab") if C else ""
    Oq = or_block(O, "tiab") if O else ""

    # minimal anchor variants (take 1–2 most specific tokens if present)
    anchors = [x for x in P if "pectus" in x or "nuss" in x or "mirpe" in x] + [x for x in I if "cryo" in x]
    anchors = anchors[:2]
    Aq = or_block(anchors, "tiab") if anchors else ""

    # PubType filter (optional)
    if proto.use_pubtype_query_filter and proto.designs_allowed:
        pt = " OR ".join(f"\"{d}\"[Publication Type]" for d in proto.designs_allowed)
        PT = f" AND ({pt})"
    else:
        PT = ""

    # Candidate set (ordered from recall→precision)
    candidates = []

    # Base PI
    if Pq and Iq:
        candidates.append({"label":"PI","expr":f"{Pq} AND {Iq}{PT}"})
        # Variants with anchors / outcomes / comparators
        if Aq:
            candidates.append({"label":"PI+A","expr":f"{Pq} AND {Iq} AND {Aq}{PT}"})
        if Cq:
            candidates.append({"label":"PI+C","expr":f"{Pq} AND {Iq} AND {Cq}{PT}"})
        if Oq:
            candidates.append({"label":"PI+O","expr":f"{Pq} AND {Iq} AND {Oq}{PT}"})
        if Cq and Oq:
            candidates.append({"label":"PI+C+O","expr":f"{Pq} AND {Iq} AND {Cq} AND {Oq}{PT}"})
    else:
        # fallback P or I alone (shouldn’t be needed if bins OK)
        if Pq: candidates.append({"label":"P","expr":f"{Pq}{PT}"})
        if Iq: candidates.append({"label":"I","expr":f"{Iq}{PT}"})

    # A compact strict version: take top-3 of each P/I to prevent over-OR
    Pq_strict = or_block(P[:3], "tiab"); Iq_strict = or_block(I[:3], "tiab")
    if Pq_strict and Iq_strict and f"{Pq_strict} AND {Iq_strict}{PT}" not in [c["expr"] for c in candidates]:
        candidates.append({"label":"PI_strict","expr":f"{Pq_strict} AND {Iq_strict}{PT}"})

    return candidates

def choose_queries_by_counts(candidates: List[Dict[str,Any]]) -> Tuple[List[Dict[str,Any]], Dict[str,int]]:
    # score by closeness to window; keep up to 5
    counts={}
    for c in candidates:
        cnt,_ = esearch_count_and_ids(c["expr"], retmax=1)
        counts[c["label"]] = cnt
        print(f"  [{c['label']}] hits={cnt}")
        time.sleep(REQUEST_DELAY)
    # prioritize ones inside window; else closest above min; else highest below
    ranked=[]
    for c in candidates:
        cnt = counts[c["label"]]
        dist = 0 if (TARGET_MIN <= cnt <= TARGET_MAX) else (abs(cnt - ((TARGET_MIN+TARGET_MAX)//2)))
        ranked.append((dist, -cnt, c))
    ranked.sort(key=lambda t: (t[0], t[1]))
    picked=[c for _,__,c in ranked[:5]]
    return picked, counts

print("Cell 4 ready.")


In [ ]:
# --- Cell 5: Drive protocol → seed mining → binning → query selection ---

# Provide your protocol: path or inline JSON string
# Example inline (edit as needed):
proto_json = """
{
  "narrative_question": "In children/adolescents undergoing Nuss/MIRPE, does intraoperative intercostal nerve cryoablation improve acute analgesia vs epidural/blocks/multimodal?",
  "deterministic_filters": {"languages":["english","spanish","portuguese"], "year_min": 2010},
  "designs_allowed": ["Randomized Controlled Trial","Comparative Study","Cohort Studies","Case-Control Studies","Observational Study"],
  "publication_types_allowlist": [],
  "publication_types_blocklist": ["Editorial","Letter","Comment","News","Case Reports"],
  "P_terms": ["pectus excavatum","nuss","mirpe","children","adolescent","pediatric"],
  "I_terms": ["intercostal nerve","cryoablation","cryoanalgesia"],
  "C_terms": ["thoracic epidural","paravertebral block","intercostal nerve block","erector spinae plane block","multimodal analgesia"],
  "O_terms": ["opioid consumption","pain scores","length of stay"],
  "mandatory_blocks": ["P","I"],
  "key_pmids": [40818798, 37364610],
  "use_pubtype_query_filter": false
}
""".strip()

proto = load_protocol(proto_json)

print("Protocol loaded.")
print("  Key PMIDs:", proto.key_pmids)
seed_recs = fetch_records_by_pmids(proto.key_pmids)
print(f"Fetched {len(seed_recs)} seed records.")
mesh_terms = mine_mesh_from_key_articles(seed_recs)
print(f"Mined {len(mesh_terms)} unique MeSH terms from seeds.")

bins = bin_mesh_terms(mesh_terms, proto)
print("Bins:")
for k in ("P","I","C","O"):
    print(f"  {k}: {bins[k]}")

cands = build_candidate_queries(bins, proto)
print(f"Built {len(cands)} candidate queries.")
for c in cands:
    print("  -", c["label"], ":", c["expr"][:140] + ("..." if len(c["expr"])>140 else ""))

picked, counts = choose_queries_by_counts(cands)
print("\nPicked queries:")
for c in picked:
    print(f"  [{c['label']}] hits={counts[c['label']]}")


In [ ]:
# --- Cell 6: Fetch, dedupe, prefilter ---

def execute_query_fetch(expr: str, budget: int) -> List[str]:
    # Fetch IDs by paging if needed (but we only need up to budget overall later)
    cnt, ids = esearch_count_and_ids(expr, retmax=min(3000, budget))
    return ids

def prefilter_record(rec: Dict[str,Any], proto: Protocol) -> Tuple[bool,str]:
    # Year
    ymin = int(proto.deterministic_filters.get("year_min", 0) or 0)
    if rec.get("year") and ymin and rec["year"] < ymin:
        return False, f"year<{ymin}"
    # Language
    langs = [l.lower() for l in (proto.deterministic_filters.get("languages") or [])]
    if rec.get("language") and langs and rec["language"].lower() not in langs:
        return False, f"lang={rec.get('language')}"
    # Publication types
    pts = set(rec.get("pubtypes") or [])
    bl = set(proto.publication_types_blocklist or [])
    if pts & bl:
        return False, f"pubtype_block={list(pts & bl)}"
    al = set(proto.publication_types_allowlist or [])
    # If allowlist given → require intersection; else let designs_allowed act as soft allow
    if al and not (pts & al):
        return False, "not_in_allowlist"
    # Designs_allowed (soft): keep if intersects; else keep as neutral
    if proto.designs_allowed:
        aliases = set(proto.designs_allowed)
        if pts & aliases:
            return True, "design_ok"
    return True, "neutral"

def fetch_universe(picked: List[Dict[str,Any]], budget_total: int = UNIVERSE_FETCH_MAX) -> List[Dict[str,Any]]:
    if not picked: return []
    per = max(1, budget_total // len(picked))
    all_ids=[]
    for c in picked:
        ids = execute_query_fetch(c["expr"], per)
        all_ids += ids
        print(f"  [{c['label']}] fetched ids n={len(ids)}")
    # Dedup and cap
    ids = list(dict.fromkeys(all_ids))[:budget_total]
    print(f"[Universe] total dedup PMIDs = {len(ids)}")
    # Fetch records
    batch=300
    recs=[]
    for i in range(0, len(ids), batch):
        xml = efetch_xml(ids[i:i+batch])
        recs += parse_pubmed_xml(xml)
    # Prefilter
    kept=[]; drops=collections.Counter()
    for r in recs:
        ok, why = prefilter_record(r, proto)
        if ok: kept.append(r)
        else: drops[why]+=1
    print(f"[Prefilter] kept={len(kept)}/{len(recs)}; drops={dict(drops)}")
    return kept

universe = fetch_universe(picked, budget_total=UNIVERSE_FETCH_MAX)
print(f"Universe size after prefilter: {len(universe)}")


In [ ]:
# --- Cell 7: Scoring & triage table ---

def build_reference_text(proto: Protocol, bins: Dict[str,List[str]]) -> str:
    parts = [proto.narrative_question]
    parts += ["; ".join(bins.get("P",[])[:6]), "; ".join(bins.get("I",[])[:6])]
    if proto.C_terms or bins.get("C"): parts += ["; ".join(bins.get("C",[])[:4])]
    if proto.O_terms or bins.get("O"): parts += ["; ".join(bins.get("O",[])[:4])]
    return " ".join(p for p in parts if p).strip()

def compute_tfidf_scores(universe: List[Dict[str,Any]], ref_text: str) -> np.ndarray:
    if not SK_TFIDF_OK:
        print("sklearn not available → TF-IDF disabled (zeros).")
        return np.zeros(len(universe), dtype=float)
    docs = [((r.get("title","") or "") + " " + (r.get("abstract","") or "")).strip() for r in universe]
    vec = TfidfVectorizer(stop_words="english", max_features=60000)
    X = vec.fit_transform(docs + [ref_text])
    ref = X[-1]
    sims = cosine_similarity(X[:-1], ref)
    return sims.ravel()

def compute_embedding_scores(universe: List[Dict[str,Any]], ref_text: str) -> np.ndarray:
    if not (USE_EMBEDDINGS and SBERT_OK):
        print("Embeddings disabled or sentence-transformers unavailable → zeros.")
        return np.zeros(len(universe), dtype=float)
    try:
        model = SentenceTransformer(EMBED_MODEL_NAME)
    except Exception as e:
        print(f"Embedding model load failed: {e} → embeddings off.")
        return np.zeros(len(universe), dtype=float)
    docs = [((r.get("title","") or "") + " " + (r.get("abstract","") or "")).strip() for r in universe]
    E = model.encode(docs, show_progress_bar=False, normalize_embeddings=True)
    q = model.encode([ref_text], show_progress_bar=False, normalize_embeddings=True)
    sims = (E @ q.T).ravel()
    return sims

# ----- iCite helpers -----

def icite_fetch(pmids: List[int]) -> Dict[int, Dict[str,Any]]:
    out={}
    pmids=[int(p) for p in pmids if p]
    if not pmids: return out
    # chunk
    for i in range(0, len(pmids), 200):
        params={"pmids":",".join(str(p) for p in pmids[i:i+200]), "legacy":"false"}
        r = requests.get(ICITE_BASE, params=params, timeout=ICITE_HTTP_TIMEOUT)
        if r.status_code != 200:
            time.sleep(ICITE_SLEEP); continue
        data = r.json().get("data", r.json())
        for rec in data:
            p = int(rec.get("pmid") or rec.get("_id"))
            out[p] = {
                "citedBy": [int(x) for x in (rec.get("citedByPmids") or rec.get("cited_by") or [])],
                "refs":    [int(x) for x in (rec.get("citedPmids")  or rec.get("references") or [])],
                "year": rec.get("year"),
                "title": rec.get("title")
            }
        time.sleep(ICITE_SLEEP)
    return out

def citation_proximity_scores(universe_pmids: List[int], seed_pmids: List[int]) -> np.ndarray:
    # count overlap in (refs ∪ citedBy) with seeds
    look = list(dict.fromkeys(universe_pmids + seed_pmids))
    meta = icite_fetch(look)
    seed_set = set(seed_pmids)
    scores=[]
    for p in universe_pmids:
        info = meta.get(int(p), {})
        neighs = set(info.get("citedBy", [])) | set(info.get("refs", []))
        score = len(neighs & seed_set)
        scores.append(float(score))
    return np.array(scores, dtype=float)

# ----- Gate: simple domain presence P & I -----

def gate_PI_presence(rec: Dict[str,Any], bins: Dict[str,List[str]]) -> bool:
    text = ((rec.get("title","") or "") + " " + (rec.get("abstract","") or "")).lower()
    P_hit = any(_normalize_token(t) in text for t in bins.get("P",[])[:8])
    I_hit = any(_normalize_token(t) in text for t in bins.get("I",[])[:8])
    return bool(P_hit and I_hit)

# ----- Assemble triage table -----

def build_triage_table(universe: List[Dict[str,Any]], proto: Protocol, bins: Dict[str,List[str]]):
    ref_text = build_reference_text(proto, bins)
    tfidf = compute_tfidf_scores(universe, ref_text)
    emb   = compute_embedding_scores(universe, ref_text)
    pmids = [int(r["pmid"]) for r in universe]
    prox  = citation_proximity_scores(pmids, proto.key_pmids) if pmids and proto.key_pmids else np.zeros(len(universe))

    # Normalize each score to [0,1]
    def norm(v):
        v = np.array(v, dtype=float)
        if len(v)==0: return v
        lo, hi = float(np.min(v)), float(np.max(v))
        if hi - lo < 1e-12: return np.zeros_like(v)
        return (v - lo) / (hi - lo)

    tfidf_n = norm(tfidf)
    emb_n   = norm(emb)
    prox_n  = norm(prox)

    # Weighted triage score (PATCH POINT)
    # If embeddings off → weight stays on tfidf and proximity
    w_tfidf, w_emb, w_prox = 0.55, 0.20, 0.25 if USE_EMBEDDINGS else (0.70, 0.00, 0.30)
    score = w_tfidf*tfidf_n + w_emb*emb_n + w_prox*prox_n

    rows=[]
    for i, r in enumerate(universe):
        rows.append({
            "pmid": int(r["pmid"]),
            "year": r.get("year"),
            "language": r.get("language"),
            "pubtypes": "; ".join(r.get("pubtypes") or []),
            "title": r.get("title","")[:500],
            "tfidf": float(tfidf_n[i]),
            "embed": float(emb_n[i]),
            "prox_to_seeds": float(prox_n[i]),
            "score": float(score[i]),
            "gate_PI": gate_PI_presence(r, bins)
        })
    rows.sort(key=lambda x: (-x["gate_PI"], -x["score"], -(x["tfidf"]+x["prox_to_seeds"])))
    return rows

triage_rows = build_triage_table(universe, proto, bins)
print(f"Triage table rows: {len(triage_rows)} (showing top 10)")
for row in triage_rows[:10]:
    print(row["pmid"], row["year"], f"gatePI={row['gate_PI']}", f"score={row['score']:.3f}", "-", row["title"][:90])


In [ ]:
# --- Cell 8: Stage-1 select & export ---

import csv

# PATCH POINT: selection thresholds
REQUIRE_PI_GATE = True
TOP_N_ABSOLUTE  = 150     # take top N after sorting
MIN_SCORE_FLOOR = 0.20    # also filter by minimal triage score

def select_stage1(tr_rows: List[Dict[str,Any]]) -> List[int]:
    rows = [r for r in tr_rows if (r["score"]>=MIN_SCORE_FLOOR and (r["gate_PI"] or not REQUIRE_PI_GATE))]
    rows = rows[:TOP_N_ABSOLUTE]
    return [int(r["pmid"]) for r in rows]

stage1_pmids = select_stage1(triage_rows)
print(f"Stage-1 selected PMIDs: {len(stage1_pmids)}")

# Export triage table
triage_csv = OUT_DIR/"triage_stage1.csv"
with open(triage_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(triage_rows[0].keys()))
    w.writeheader()
    for r in triage_rows:
        w.writerow(r)
print("Wrote:", triage_csv)

# Export stage-1 list
(stage1_list_path := OUT_DIR/"stage1_pmids.json").write_text(json.dumps(stage1_pmids, indent=2), encoding="utf-8")
print("Wrote:", stage1_list_path)

# Export a downloader handoff CSV (PMID, Year, FirstAuthor, Title, DOI placeholder)
# We'll fetch DOIs now to fill this.
def fetch_dois_for_pmids(pmids: List[int]) -> Dict[int, Optional[str]]:
    out={int(p):None for p in pmids}
    batch=200
    for i in range(0, len(pmids), batch):
        xml = efetch_xml([str(x) for x in pmids[i:i+batch]])
        root = ET.fromstring(xml)
        for art in root.findall(".//PubmedArticle"):
            pmid = art.findtext(".//PMID") or ""
            pmid = int(pmid) if pmid.isdigit() else None
            if pmid is None: continue
            doi = None
            n = art.find(".//ArticleIdList/ArticleId[@IdType='doi']")
            if n is None:
                n = art.find(".//MedlineCitation/Article/ELocationID[@EIdType='doi'][@ValidYN='Y']")
            if n is not None and n.text:
                doi = n.text.strip()
            out[pmid] = doi
        time.sleep(REQUEST_DELAY)
    return out

stage1_recs = fetch_records_by_pmids(stage1_pmids)
pmid_to_doi = fetch_dois_for_pmids(stage1_pmids)

downloader_csv = OUT_DIR/"stage1_fulltext_handoff.csv"
with open(downloader_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["PMID","Year","FirstAuthor","Title","DOI"])
    by_id = {int(r["pmid"]): r for r in stage1_recs}
    for p in stage1_pmids:
        r = by_id.get(int(p), {})
        w.writerow([p, r.get("year",""), r.get("first_author",""), (r.get("title","") or "").replace("\n"," "), pmid_to_doi.get(int(p)) or ""])
print("Wrote handoff CSV for your PDF retriever:", downloader_csv)


In [ ]:
# --- Cell 9: Snowball from Stage-1 includes; build Stage-2 universe ---

def icite_neighbors_batch(pmids: List[int]) -> Set[int]:
    meta = icite_fetch(pmids)
    nbrs=set()
    for p,info in meta.items():
        nbrs |= set(info.get("citedBy",[]))
        nbrs |= set(info.get("refs",[]))
    return nbrs

universe_pmids_set = {int(r["pmid"]) for r in universe}
stage1_set = set(stage1_pmids)

snowball = icite_neighbors_batch(stage1_pmids)
new_pmids = list(sorted((snowball - universe_pmids_set)))
print(f"Snowball neighbors: {len(snowball)} | New (not in universe): {len(new_pmids)}")

# Fetch & prefilter these new PMIDs
stage2_recs = fetch_records_by_pmids(new_pmids)
stage2_keep=[]
for r in stage2_recs:
    ok,_ = prefilter_record(r, proto)
    if ok: stage2_keep.append(r)
print(f"Stage-2 pool after prefilter: {len(stage2_keep)}")

# Score and select stage-2
triage2 = build_triage_table(stage2_keep, proto, bins)
print(f"Triage-2 rows: {len(triage2)} (showing top 10)")
for row in triage2[:10]:
    print(row["pmid"], row["year"], f"gatePI={row['gate_PI']}", f"score={row['score']:.3f}", "-", row["title"][:90])

stage2_pmids = select_stage1(triage2)  # reuse same selector
print(f"Stage-2 selected PMIDs: {len(stage2_pmids)}")

# Exports
import csv, json
triage2_csv = OUT_DIR/"triage_stage2.csv"
with open(triage2_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(triage2[0].keys()) if triage2 else ["pmid"])
    if triage2: w.writeheader()
    for r in triage2:
        w.writerow(r)
print("Wrote:", triage2_csv)

(stage2_list_path := OUT_DIR/"stage2_pmids.json").write_text(json.dumps(stage2_pmids, indent=2), encoding="utf-8")
print("Wrote:", stage2_list_path)

# Handoff for full-text retrieval (stage-2)
stage2_dois = fetch_dois_for_pmids(stage2_pmids)
stage2_recs_full = fetch_records_by_pmids(stage2_pmids)
downloader2_csv = OUT_DIR/"stage2_fulltext_handoff.csv"
with open(downloader2_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["PMID","Year","FirstAuthor","Title","DOI"])
    by_id = {int(r["pmid"]): r for r in stage2_recs_full}
    for p in stage2_pmids:
        r = by_id.get(int(p), {})
        w.writerow([p, r.get("year",""), r.get("first_author",""), (r.get("title","") or "").replace("\n"," "), stage2_dois.get(int(p)) or ""])
print("Wrote stage-2 handoff CSV:", downloader2_csv)


In [ ]:
# === Cell A — BUILD FINAL HANDOFF (single full-text stage) ===
# Consolidate final INCLUDED set (post-snowball, title/abstract only) into one CSV for full-text fetch.

import pathlib, pandas as pd

OUT_DIR = pathlib.Path("triage_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# These are the files your earlier abstract-only pipeline should have written.
# If your filenames differ, update paths here — nothing else in the notebook needs changing.
STAGE1_INCLUDED = OUT_DIR / "stage1_included.csv"    # title/abstract screened includes
STAGE2_INCLUDED = OUT_DIR / "stage2_included.csv"    # snowball includes (title/abstract)
FINAL_HANDOFF   = OUT_DIR / "final_fulltext_handoff.csv"

def _read_opt(path):
    if path.exists():
        df = pd.read_csv(path)
        # Normalize required columns if present under slightly different names
        colmap = {
            'pmid':'PMID', 'PMId':'PMID', 'Pmid':'PMID',
            'doi':'DOI', 'Doi':'DOI',
            'title':'Title', 'Title':'Title',
            'year':'Year', 'Year':'Year',
            'first_author':'FirstAuthor', 'First Author':'FirstAuthor', 'FirstAuthor':'FirstAuthor'
        }
        for c in list(df.columns):
            if c in colmap and colmap[c] not in df.columns:
                df[colmap[c]] = df[c]
        return df
    return pd.DataFrame(columns=['PMID','DOI','Title','Year','FirstAuthor'])

d1 = _read_opt(STAGE1_INCLUDED)
d2 = _read_opt(STAGE2_INCLUDED)

# union + dedupe by PMID (string)
for df in (d1, d2):
    if 'PMID' in df.columns:
        df['PMID'] = df['PMID'].astype(str).str.replace(r'\.0$','', regex=True).str.strip()

final_df = pd.concat([d1, d2], ignore_index=True)
if not final_df.empty:
    final_df = final_df.drop_duplicates(subset=['PMID'], keep='first')

# Keep canonical columns the fetcher needs; fill if missing
for col in ['PMID','DOI','Title','Year','FirstAuthor']:
    if col not in final_df.columns:
        final_df[col] = ""

# Sort (optional): newest first, then PMID
with pd.option_context('mode.use_inf_as_na', True):
    if 'Year' in final_df.columns:
        # coerce Year to numeric for sort; non-numeric to NaN (sort to end)
        final_df['Year_num'] = pd.to_numeric(final_df['Year'], errors='coerce')
        final_df = final_df.sort_values(['Year_num','PMID'], ascending=[False, True]).drop(columns=['Year_num'])
    else:
        final_df = final_df.sort_values('PMID')

final_df.to_csv(FINAL_HANDOFF, index=False)
print(f"[BUILD HANDOFF] final_included rows = {len(final_df)}")
print(f"[BUILD HANDOFF] wrote {FINAL_HANDOFF}")


In [ ]:
# === Cell B — FULL-TEXT FETCH (Final) — FUNCTION MODE ===
# INPUT : triage_out/final_fulltext_handoff.csv
# OUTPUT: triage_out/fulltext_failed_pmids.txt (append)

import os, re, pathlib, pandas as pd

OUT_DIR = pathlib.Path("triage_out")
FINAL_HANDOFF = OUT_DIR / "final_fulltext_handoff.csv"
FAILED_FILE   = OUT_DIR / "fulltext_failed_pmids.txt"
PDF_OUTDIR    = pathlib.Path("downloaded_pdfs_final")
PDF_OUTDIR.mkdir(parents=True, exist_ok=True)

assert FINAL_HANDOFF.exists(), f"Missing {FINAL_HANDOFF}. Run Cell A first."
df = pd.read_csv(FINAL_HANDOFF)

# ---------- PASTE YOUR FETCHER BELOW ----------
# (Your full script, unchanged except the bottom __main__ block is commented out.)

import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse
from urllib3.util.retry import Retry
import xml.etree.ElementTree as ET
from requests.adapters import HTTPAdapter
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import random
from collections import Counter

# --- Configuration ---
EXCEL_FILE_PATH = r"C:\Users\Galaxy\Downloads\screening_ERAS.xlsx"
OUTPUT_PDF_DIR = "downloaded_pdfs_v4"

SCI_HUB_DOMAINS = [  # Prioritize historically better ones
    "https://sci-hub.se",
    "https://sci-hub.ru",
    "https://sci-hub.ren",
    "https://sci-hub.wf",  # Problematic in logs, but kept for test diversity
    "https://sci-hub.ee",
    "https://sci-hub.st",  # Problematic in logs
]
REQUEST_DELAY_SCIHUB = 1.5  # Delay between domain attempts for *same* article
MAX_WORKERS_SCIHUB = 4
PREFERRED_DOMAINS_COUNT = 1  # Try to find one solid preferred domain
FALLBACK_DOMAINS_COUNT = 2
LAST_SUCCESSFUL_PARSING_DOMAIN = None  # Global to hold the last domain that allowed a successful parse

NCBI_API_KEY = "YOUR_API_KEY_HERE"
CROSSREF_MAILTO = "your_email@example.com"
EFETCH_BATCH_SIZE = 100
REQUEST_DELAY_NCBI = 0.35

# --- Logging ---
# Avoid duplicate handlers on re-run in notebooks
log_formatter = logging.Formatter('%(asctime)s - %(levelname)s - [%(threadName)s] - %(message)s')
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.handlers.clear()  # <<< important in notebooks

ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(log_formatter)
logger.addHandler(ch)
# File handler (optional) – keep commented if not needed
# fh = logging.FileHandler('pdf_downloader.log', mode='w')
# fh.setLevel(logging.DEBUG)
# fh.setFormatter(log_formatter)
# logger.addHandler(fh)

# --- NCBI Session & Helper ---
session_ncbi = requests.Session()
retries_ncbi = Retry(total=3, backoff_factor=0.5, status_forcelist=[500, 502, 503, 504],
                     allowed_methods=frozenset(['GET', 'POST']))
adapter_ncbi = HTTPAdapter(pool_connections=10, pool_maxsize=10, max_retries=retries_ncbi)
session_ncbi.mount("https://", adapter_ncbi)
session_ncbi.mount("http://", adapter_ncbi)
ncbi_user_agent_base = 'SciHubPDFDownloader/2.3 (Python Requests; mailto:{})'

def _get_ncbi_params(extra=None):
    params = {"tool": "pdf_downloader_eras_v4", "email": CROSSREF_MAILTO}
    if NCBI_API_KEY and NCBI_API_KEY != "YOUR_API_KEY_HERE":
        params["api_key"] = NCBI_API_KEY
    if CROSSREF_MAILTO == "your_email@example.com" and "email" in params:
        logger.warning("`CROSSREF_MAILTO` is placeholder.")
        params.pop("email", None)
    if extra:
        params.update(extra)
    return params

def fetch_dois_for_pmids_batched(pmids_list, batch_size=EFETCH_BATCH_SIZE):
    t_start_fetch_doi = time.time()
    pmid_to_doi_map = {pmid: None for pmid in pmids_list}
    valid_pmids_to_query = [pmid for pmid in pmids_list if pmid]
    if not valid_pmids_to_query:
        return pmid_to_doi_map
    logger.info(f"Fetching DOIs for {len(valid_pmids_to_query)} PMIDs from NCBI (batch: {batch_size})")
    for i in range(0, len(valid_pmids_to_query), batch_size):
        batch_pmids = valid_pmids_to_query[i:i + batch_size]
        logger.debug(f"  NCBI EFetch batch {i//batch_size + 1} ({len(batch_pmids)} PMIDs)")
        post_data = _get_ncbi_params({"db": "pubmed", "retmode": "xml", "id": ",".join(batch_pmids)})
        try:
            response = session_ncbi.post("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
                                         data=post_data, timeout=60)
            response.raise_for_status()
            if not response.content:
                logger.warning(f"NCBI EFetch batch {i//batch_size+1} empty content.")
                continue
            root = ET.fromstring(response.content)
            for art_node in root.findall(".//PubmedArticle"):
                pmid_elem = art_node.find(".//MedlineCitation/PMID")
                pmid = pmid_elem.text if pmid_elem is not None else None
                if not pmid:
                    continue
                doi = None
                doi_elem = art_node.find(f".//ArticleIdList/ArticleId[@IdType='doi']")
                if doi_elem is None:
                    doi_elem = art_node.find(f".//MedlineCitation/Article/ELocationID[@EIdType='doi'][@ValidYN='Y']")
                if doi_elem is not None and doi_elem.text:
                    doi = doi_elem.text.strip()
                if pmid in pmid_to_doi_map and doi:
                    pmid_to_doi_map[pmid] = doi
        except Exception as e:
            logger.error(f"NCBI EFetch batch {i//batch_size+1} error: {e}", exc_info=False)
        time.sleep(REQUEST_DELAY_NCBI if not (NCBI_API_KEY and NCBI_API_KEY != "YOUR_API_KEY_HERE") else 0.11)
    found_doi_count = sum(1 for d_val in pmid_to_doi_map.values() if d_val)
    logger.info(f"DOI fetching complete. Found for {found_doi_count}/{len(valid_pmids_to_query)} PMIDs. "
                f"(Took {time.time() - t_start_fetch_doi:.2f}s)")
    return pmid_to_doi_map

# --- Sci-Hub Session & Helpers ---
session_scihub = requests.Session()
scihub_headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/100.0.4896.127 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
    'Accept-Language': 'en-US,en;q=0.9', 'DNT': '1', 'Upgrade-Insecure-Requests': '1'
}
session_scihub.headers.update(scihub_headers)
retries_scihub = Retry(total=1, backoff_factor=0.2, status_forcelist=[500, 502, 503, 504],
                       allowed_methods=frozenset(['GET']))
adapter_scihub = HTTPAdapter(pool_connections=MAX_WORKERS_SCIHUB + 5,
                             pool_maxsize=MAX_WORKERS_SCIHUB + 5,
                             max_retries=retries_scihub)
session_scihub.mount("https://", adapter_scihub)
session_scihub.mount("http://", adapter_scihub)

def sanitize_filename(filename):
    filename = re.sub(r'[\\/*?:"<>|]', "", filename)
    filename = filename.replace(" ", "_").replace("__", "_")
    return filename[:200]

def find_pdf_url_on_scihub_page(html_content, scihub_page_url):
    soup = BeautifulSoup(html_content, 'html.parser')
    pdf_url = None
    scihub_domain_base = f"{urlparse(scihub_page_url).scheme}://{urlparse(scihub_page_url).netloc}"
    logger.debug(f"Parsing HTML from {scihub_page_url} (base: {scihub_domain_base})")

    iframe = soup.find('iframe', id='pdf')
    if iframe and iframe.get('src'):
        pdf_url = iframe['src']
    elif not pdf_url:
        iframe = soup.find('iframe', id='article')
        if iframe and iframe.get('src'):
            pdf_url = iframe['src']
    if not pdf_url:
        for iframe_tag in soup.find_all('iframe'):
            src_val = iframe_tag.get('src')
            if src_val and ('.pdf' in src_val or 'downloads/' in src_val or 'tree/' in src_val):
                pdf_url = src_val
                break
    if not pdf_url:
        embed = soup.find('embed', type='application/pdf')
        if embed and embed.get('src'):
            pdf_url = embed['src']
    if not pdf_url:
        selectors = [
            'button#download', 'a#download', 'div#download',
            'button[onclick*="location.href"]', 'a[onclick*="location.href"]', 'div[onclick*="location.href"]',
            'a[href*=".pdf"]', 'a[href*="downloads/"]', 'a[href*="/tree/"]',
        ]
        for selector in selectors:
            element = soup.select_one(selector)
            if not element:
                continue
            potential_url = None
            if element.get('onclick'):
                m = re.search(r"location\.href\s*=\s*['\"]([^'\"]+)['\"]", element['onclick'])
                if m:
                    potential_url = m.group(1)
            elif element.get('href') and element.get('href') != '#':
                potential_url = element['href']
            if potential_url:
                pdf_url = potential_url
                break
    if pdf_url:
        if pdf_url.startswith("//"):
            pdf_url = urlparse(scihub_page_url).scheme + ":" + pdf_url
        elif not pdf_url.startswith("http"):
            pdf_url = urljoin(scihub_domain_base, pdf_url)
        return pdf_url
    logger.warning(f"  [!] Could not find PDF link on page {scihub_page_url} using defined selectors.")
    return None

def test_scihub_domain(domain, test_identifier="10.1186/1471-2121-7-27"):
    test_url = f"{domain.strip('/')}/{test_identifier}"
    try:
        response = session_scihub.get(test_url, timeout=7)
        if response.status_code == 200 and 'text/html' in response.headers.get('Content-Type','').lower():
            if (find_pdf_url_on_scihub_page(response.content, response.url)
                or "article not found" in response.text.lower()
                or "sci-hub" in response.text.lower()):
                logger.info(f"  [Domain Test] {domain} PASSED")
                return True
            logger.warning(f"  [Domain Test] {domain} FAILED (unrecognized structure)")
            return False
        if response.status_code == 404 and "article not found" in response.text.lower():
            logger.info(f"  [Domain Test] {domain} PASSED (404 acceptable)")
            return True
        logger.warning(f"  [Domain Test] {domain} FAILED. Status: {response.status_code}, "
                       f"Content-Type: {response.headers.get('Content-Type','')}")
    except requests.exceptions.RequestException as e:
        logger.warning(f"  [Domain Test] {domain} FAILED request: {e}")
    return False

def initialize_working_domains():
    global PREFERRED_DOMAINS_COUNT, LAST_SUCCESSFUL_PARSING_DOMAIN
    working_domains = []
    logger.info("Initializing working Sci-Hub domains by testing...")
    shuffled = random.sample(SCI_HUB_DOMAINS, len(SCI_HUB_DOMAINS))
    for domain in shuffled:
        if test_scihub_domain(domain):
            working_domains.append(domain)
            if len(working_domains) >= PREFERRED_DOMAINS_COUNT:
                break
    if working_domains:
        logger.info(f"Working Sci-Hub domains: {working_domains}")
        LAST_SUCCESSFUL_PARSING_DOMAIN = working_domains[0]
    else:
        logger.error("CRITICAL: Could not identify any working Sci-Hub domains.")
    return working_domains

def download_pdf_from_scihub_attempt(identifier, identifier_type, pmid_for_filename,
                                     first_author, year, output_dir, available_domains):
    global LAST_SUCCESSFUL_PARSING_DOMAIN
    base_filename = f"{year}_{first_author}_{pmid_for_filename}"
    pdf_filename = sanitize_filename(base_filename) + ".pdf"
    pdf_filepath = os.path.join(output_dir, pdf_filename)

    if os.path.exists(pdf_filepath):
        return True, "Already Exists"

    domains_to_try_ordered = []
    if LAST_SUCCESSFUL_PARSING_DOMAIN and LAST_SUCCESSFUL_PARSING_DOMAIN in available_domains:
        domains_to_try_ordered.append(LAST_SUCCESSFUL_PARSING_DOMAIN)
    for d in available_domains:
        if d not in domains_to_try_ordered:
            domains_to_try_ordered.append(d)
    if not domains_to_try_ordered:
        remaining = [d for d in SCI_HUB_DOMAINS if d not in domains_to_try_ordered]
        random.shuffle(remaining)
        domains_to_try_ordered.extend(remaining[:FALLBACK_DOMAINS_COUNT + 1])
    domains_to_try_ordered = list(dict.fromkeys(domains_to_try_ordered))

    logger.debug(f"  PMID {pmid_for_filename}: Trying {identifier_type} '{identifier}'. Order: {domains_to_try_ordered}")

    for scihub_domain in domains_to_try_ordered:
        search_url = f"{scihub_domain.strip('/')}/{identifier}"
        logger.debug(f"    PMID {pmid_for_filename}: Attempting {search_url}")
        try:
            time.sleep(REQUEST_DELAY_SCIHUB)
            response = session_scihub.get(search_url, timeout=20)
            response.raise_for_status()
            content_type = response.headers.get('Content-Type', '').lower()

            if 'application/pdf' in content_type:
                logger.info(f"    PMID {pmid_for_filename}: Direct PDF from {search_url}!")
                with open(pdf_filepath, 'wb') as f:
                    f.write(response.content)
                LAST_SUCCESSFUL_PARSING_DOMAIN = scihub_domain
                return True, scihub_domain

            if 'text/html' in content_type:
                direct_pdf_url = find_pdf_url_on_scihub_page(response.content, response.url)
                if direct_pdf_url:
                    logger.info(f"    PMID {pmid_for_filename}: Found PDF URL: {direct_pdf_url}. Downloading...")
                    time.sleep(0.5)
                    pdf_response = session_scihub.get(direct_pdf_url, timeout=45, stream=True)
                    pdf_response.raise_for_status()
                    if 'application/pdf' in pdf_response.headers.get('Content-Type','').lower():
                        with open(pdf_filepath, 'wb') as f:
                            for chunk in pdf_response.iter_content(chunk_size=8192):
                                f.write(chunk)
                        LAST_SUCCESSFUL_PARSING_DOMAIN = scihub_domain
                        return True, scihub_domain
                    logger.warning(f"    PMID {pmid_for_filename}: Link not PDF. Type: {pdf_response.headers.get('Content-Type','')}")
            else:
                logger.warning(f"    PMID {pmid_for_filename}: Unexpected Content-Type '{content_type}' from {search_url}.")
        except requests.exceptions.HTTPError as e:
            if e.response is not None:
                code = e.response.status_code
                if code == 404:
                    logger.debug(f"    PMID {pmid_for_filename}: 404 on {search_url}.")
                elif code == 403:
                    logger.warning(f"    PMID {pmid_for_filename}: 403 Forbidden on {search_url}.")
                else:
                    logger.warning(f"    PMID {pmid_for_filename}: HTTP Error {code} on {search_url}.")
        except requests.exceptions.RequestException as e:
            logger.warning(f"    PMID {pmid_for_filename}: Request failed for {search_url}: {e}", exc_info=False)

    return False, None

def process_article_row(args):
    index, row, pmid_to_doi_map, output_dir, working_domains_for_run = args
    pmid_str = str(row.get('PMID', '')).strip().split('.')[0]
    if not pmid_str:
        return pmid_str, False, "No PMID", None

    first_author = str(row.get('First Author', 'UnknownAuthor')).strip()
    year = str(row.get('Year', 'UnknownYear')).strip()

    article_log_prefix = f"Article (PMID {pmid_str}, {year}, {first_author}, ExcelRow {index+2})"
    logger.info(f"{article_log_prefix}: Starting process.")

    pdf_filepath = os.path.join(output_dir, sanitize_filename(f"{year}_{first_author}_{pmid_str}") + ".pdf")
    if os.path.exists(pdf_filepath):
        logger.info(f"{article_log_prefix}: PDF already exists.")
        return pmid_str, True, "Already Existed", None

    article_doi = pmid_to_doi_map.get(pmid_str)
    download_successful = False
    used_domain = None
    method = ""

    if article_doi:
        logger.info(f"{article_log_prefix}: Has DOI {article_doi}. Trying DOI first.")
        download_successful, used_domain = download_pdf_from_scihub_attempt(
            article_doi, "DOI", pmid_str, first_author, year, output_dir, working_domains_for_run)
        if download_successful:
            method = f"DOI via {used_domain}" if used_domain != "Already Exists" else "DOI (Exists)"
        else:
            logger.warning(f"{article_log_prefix}: DOI download failed. Fallback to PMID.")
    else:
        logger.info(f"{article_log_prefix}: No DOI found. Trying PMID directly.")

    if not download_successful:
        download_successful, used_domain = download_pdf_from_scihub_attempt(
            pmid_str, "PMID", pmid_str, first_author, year, output_dir, working_domains_for_run)
        if download_successful:
            method = f"PMID via {used_domain}" if used_domain != "Already Exists" else "PMID (Exists)"

    if download_successful:
        logger.info(f"{article_log_prefix}: SUCCESS ({method}).")
    else:
        logger.error(f"{article_log_prefix}: FAILED all attempts.")
    return pmid_str, download_successful, method, used_domain if download_successful and used_domain != "Already Exists" else None

def main():
    run_start_time = time.time()
    logger.info(f"Starting PDF download process (v4)...")
    if NCBI_API_KEY == "YOUR_API_KEY_HERE":
        logger.warning("NCBI_API_KEY is placeholder.")
    if CROSSREF_MAILTO == "your_email@example.com":
        logger.warning("CROSSREF_MAILTO is placeholder.")

    session_ncbi.headers.update({'User-Agent': ncbi_user_agent_base.format(
        CROSSREF_MAILTO if CROSSREF_MAILTO != "your_email@example.com" else "anonymous_user")})

    try:
        df = pd.read_excel(EXCEL_FILE_PATH)
    except FileNotFoundError:
        logger.error(f"Excel file not found: {EXCEL_FILE_PATH}.")
        return
    except Exception as e:
        logger.error(f"Could not read Excel: {e}", exc_info=True)
        return

    if 'PMID' not in df.columns:
        logger.error(f"'PMID' column not found. Cols: {df.columns.tolist()}")
        return
    if not os.path.exists(OUTPUT_PDF_DIR):
        os.makedirs(OUTPUT_PDF_DIR)
        logger.info(f"Created output dir: {OUTPUT_PDF_DIR}")

    all_pmids_from_excel = [str(pmid).split('.')[0] for pmid in df['PMID'].dropna().unique() if str(pmid).strip()]
    pmid_to_doi_map = fetch_dois_for_pmids_batched(all_pmids_from_excel)

    working_domains_for_this_run = initialize_working_domains()
    if not working_domains_for_this_run:
        logger.warning("No working Sci-Hub domains identified in pre-check. Using full list as fallback for each item.")

    download_stats = Counter()
    failed_pmids = []
    domain_success_counts = Counter()

    tasks = [(index, row, pmid_to_doi_map, OUTPUT_PDF_DIR, working_domains_for_this_run) for index, row in df.iterrows()]

    logger.info(f"Starting Parallel PDF Download (Workers: {MAX_WORKERS_SCIHUB}, Initial Working Domains: {working_domains_for_this_run})...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS_SCIHUB, thread_name_prefix='SciHubWorker') as executor:
        future_to_info = {executor.submit(process_article_row, task_args): (task_args[0], task_args[1].get('PMID')) for task_args in tasks}
        for future in as_completed(future_to_info):
            _excel_idx, _orig_pmid = future_to_info[future]
            try:
                _pmid_proc, success, method_msg, domain_used = future.result()
                if success:
                    if method_msg == "Already Existed":
                        download_stats['existed'] += 1
                    else:
                        download_stats['success'] += 1
                    if domain_used:
                        domain_success_counts[domain_used] += 1
                else:
                    download_stats['failure'] += 1
                    failed_pmids.append(_pmid_proc or _orig_pmid)
            except Exception as exc:
                logger.error(f"Article (Excel idx {_excel_idx}, PMID {_orig_pmid}) generated exception: {exc}", exc_info=True)
                download_stats['failure'] += 1
                failed_pmids.append(_orig_pmid)

    run_end_time = time.time()
    logger.info("--- Download Summary ---")
    logger.info(f"Total articles in Excel: {len(df)}")
    logger.info(f"Successfully downloaded: {download_stats['success']}")
    logger.info(f"Already existed:       {download_stats['existed']}")
    logger.info(f"Failed to download:    {download_stats['failure']}")
    if failed_pmids:
        logger.info(f"Failed PMIDs: {', '.join(filter(None,failed_pmids))}")
    if domain_success_counts:
        logger.info("Successes by domain:")
        for domain, count in domain_success_counts.items():
            logger.info(f"  {domain}: {count}")
    logger.info(f"PDFs saved to: {os.path.abspath(OUTPUT_PDF_DIR)}")
    logger.info(f"Total script execution time: {run_end_time - run_start_time:.2f} seconds.")
    # Optional: return list of failures from inside the script
    return failed_pmids

# (No __main__ auto-run in the cell)
# ---------- END YOUR FETCHER ----------

def fetch_fulltexts(df: pd.DataFrame, pdf_outdir: pathlib.Path) -> list[str]:
    """
    Adapter: runs YOUR fetcher on triage_out/final_fulltext_handoff.csv
    and returns a list of failed PMIDs (strings).
    """
    # Normalize columns for your script
    df2 = df.copy()
    df2["PMID"] = df2["PMID"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    if "First Author" not in df2.columns and "FirstAuthor" in df2.columns:
        df2 = df2.rename(columns={"FirstAuthor": "First Author"})
    for col in ["PMID", "Year", "First Author", "Title", "DOI"]:
        if col not in df2.columns:
            df2[col] = ""

    # Write a temp Excel your script reads
    tmp_xlsx = OUT_DIR / "_handoff_for_fetcher.xlsx"
    # Ensure an Excel writer exists
    try:
        df2.to_excel(tmp_xlsx, index=False)  # uses openpyxl if present
    except Exception as e:
        # Try xlsxwriter explicitly
        try:
            with pd.ExcelWriter(tmp_xlsx, engine="xlsxwriter") as wr:
                df2.to_excel(wr, index=False)
        except Exception:
            raise RuntimeError(
                f"Failed to write Excel handoff at {tmp_xlsx}. "
                f"Install an Excel writer: `pip install openpyxl` or `pip install xlsxwriter`.\nOriginal error: {e}"
            )

    # Set the globals your script reads
    global EXCEL_FILE_PATH, OUTPUT_PDF_DIR
    EXCEL_FILE_PATH = str(tmp_xlsx)
    OUTPUT_PDF_DIR  = str(pdf_outdir)

    # Optional: set your tokens here (or rely on your script’s defaults)
    # global NCBI_API_KEY, CROSSREF_MAILTO
    # NCBI_API_KEY = "YOUR_REAL_KEY"
    # CROSSREF_MAILTO = "you@domain.tld"

    # Run your script’s main()
    ret = None
    try:
        ret = main()
    except TypeError:
        # If you later extend main(return_failed=True)
        ret = main(return_failed=True)

    # Use your own sanitizer if present
    _sanitize = sanitize_filename if "sanitize_filename" in globals() else (
        lambda s: re.sub(r'[\\/*?:"<>|]', "", s).replace(" ", "_").replace("__", "_")[:200]
    )

    # Compute failures by checking expected filenames in OUTPUT_PDF_DIR
    present = set(os.listdir(pdf_outdir)) if os.path.isdir(pdf_outdir) else set()
    failed = []
    for _, row in df2.iterrows():
        pmid = str(row["PMID"]).strip().split(".")[0]
        year = str(row.get("Year", "") or "UnknownYear")
        first_author = str(row.get("First Author", "") or "UnknownAuthor")
        expected = _sanitize(f"{year}_{first_author}_{pmid}") + ".pdf"
        if expected not in present:
            failed.append(pmid)

    # Merge with any explicit failure list returned by your main()
    if isinstance(ret, (list, tuple, set)):
        for x in ret:
            sx = str(x).strip().split(".")[0]
            if sx and sx not in failed:
                failed.append(sx)

    return sorted(set(failed))


failed_pmids = fetch_fulltexts(df, PDF_OUTDIR)

with open(FAILED_FILE, "a", encoding="utf-8") as f:
    for p in failed_pmids:
        p = str(p).strip().split('.')[0]
        if p:
            f.write(p + "\n")

print(f"[FETCH] input rows = {len(df)}")
print(f"[FETCH] failures   = {len(failed_pmids)} → {FAILED_FILE}")
print(f"[FETCH] PDFs dir   = {PDF_OUTDIR.resolve()}")


In [ ]:
# --- Cell 10: Final merge & pack ---

final_pmids = sorted(list(set(stage1_pmids) | set(stage2_pmids)))
print(f"Final triage set: {len(final_pmids)} PMIDs")

(final_list_path := OUT_DIR/"final_triage_pmids.json").write_text(json.dumps(final_pmids, indent=2), encoding="utf-8")
print("Wrote:", final_list_path)

# Pack a master CSV with triage scores (if present)
# Merge triage rows by pmid → prefer stage-1 rows; fill with stage-2 where missing
rows_by_pmid = {int(r["pmid"]): r for r in triage_rows}
for r in triage2:
    p = int(r["pmid"])
    if p not in rows_by_pmid:
        rows_by_pmid[p] = r

master_csv = OUT_DIR/"triage_master.csv"
import csv
with open(master_csv, "w", newline="", encoding="utf-8") as f:
    cols = ["pmid","year","language","pubtypes","title","tfidf","embed","prox_to_seeds","score","gate_PI"]
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    for p in final_pmids:
        r = rows_by_pmid.get(int(p), {"pmid":p})
        w.writerow({k: r.get(k,"") for k in cols})
print("Wrote:", master_csv)


In [ ]:
# --- Cell 11: Post-download intake of failed PMIDs (manual retrieval list) ---

failed_list_file = OUT_DIR/"fulltext_failed_pmids.txt"  # one PMID per line
failed_pmids = []
if failed_list_file.exists():
    for line in failed_list_file.read_text(encoding="utf-8").splitlines():
        line=line.strip()
        if line.isdigit():
            failed_pmids.append(int(line))
failed_pmids = sorted(list(set(failed_pmids)))
print("Failed full-text PMIDs from your downloader:", len(failed_pmids))

if failed_pmids:
    recs = fetch_records_by_pmids(failed_pmids)
    dois = fetch_dois_for_pmids(failed_pmids)
    todo_csv = OUT_DIR/"manual_fulltext_todo.csv"
    import csv
    with open(todo_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["PMID","Year","FirstAuthor","Title","DOI"])
        by_id = {int(r["pmid"]): r for r in recs}
        for p in failed_pmids:
            r = by_id.get(int(p), {})
            w.writerow([p, r.get("year",""), r.get("first_author",""), (r.get("title","") or "").replace("\n"," "), dois.get(int(p)) or ""])
    print("Wrote manual full-text TODO:", todo_csv)
else:
    print("No failed PMIDs file found yet or empty.")


In [ ]:
# --- Cell 12: Summary ---

print("SUMMARY")
print("-------")
print("Seeds (key_pmids):", len(proto.key_pmids))
print("Universe (prefilter):", len(universe))
print("Stage-1 selected:", len(stage1_pmids))
print("Snowball new:", "see Cell 9 output")
print("Stage-2 selected:", len(stage2_pmids))
print("Final triage set:", len(final_pmids))
print()
print("Artifacts:")
for p in [
    OUT_DIR/"triage_stage1.csv",
    OUT_DIR/"stage1_pmids.json",
    OUT_DIR/"stage1_fulltext_handoff.csv",
    OUT_DIR/"triage_stage2.csv",
    OUT_DIR/"stage2_pmids.json",
    OUT_DIR/"stage2_fulltext_handoff.csv",
    OUT_DIR/"triage_master.csv",
    OUT_DIR/"final_triage_pmids.json",
    OUT_DIR/"fulltext_failed_pmids.txt",  # you create after your downloader run
]:
    print(" -", p)
print("\nNext steps:")
print(" 1) Run *your* full-text retrieval on the two *_handoff.csv files.")
print(" 2) Create 'triage_out/fulltext_failed_pmids.txt' with the PMIDs that still failed.")
print(" 3) Re-run Cell 11 to get 'manual_fulltext_todo.csv'.")
